In [1]:
import pandas as pd
from pathlib import Path

print(Path.cwd())

/Users/dafyddjones/projects/heart-disease-analysis/notebooks


In [2]:
data_path = Path("../data/raw/processed.cleveland.data")

print("File found:", data_path.exists())

File found: True


In [3]:
columns = [
    "age", "sex", "chest_pain_type",
    "resting_blood_pressure", "cholesterol",
    "fasting_blood_sugar", "resting_ecg",
    "max_heart_rate", "exercise_induced_angina",
    "st_depression", "st_slope", "major_vessels",
    "thal", "disease_status"
]

df = pd.read_csv(
    data_path,
    header=None,
    names=columns,
    na_values="?"
)

print("Dataset shape:", df.shape)

Dataset shape: (303, 14)


In [4]:
X = df.drop(columns="disease_status")
y = (df["disease_status"] > 0).astype(int)

print("Input shape:", X.shape)
print("Outcome counts:")
print(y.value_counts().sort_index())

Input shape: (303, 13)
Outcome counts:
disease_status
0    164
1    139
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Test records:", len(X_test))

Training records: 242
Test records: 61


In [6]:
numeric_features = [
    "age",
    "resting_blood_pressure",
    "cholesterol",
    "max_heart_rate",
    "st_depression",
    "major_vessels"
]

categorical_features = [
    "sex",
    "chest_pain_type",
    "fasting_blood_sugar",
    "resting_ecg",
    "exercise_induced_angina",
    "st_slope",
    "thal"
]

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [8]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [9]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [10]:
from sklearn.dummy import DummyClassifier

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [12]:
baseline_scores = cross_validate(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "recall", "roc_auc"]
)

pd.DataFrame(baseline_scores)[
    ["test_accuracy", "test_recall", "test_roc_auc"]
].mean().round(3)

test_accuracy    0.541
test_recall      0.000
test_roc_auc     0.500
dtype: float64

In [13]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

In [14]:
logistic_scores = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "recall", "roc_auc"]
)

pd.DataFrame(logistic_scores)[
    ["test_accuracy", "test_recall", "test_roc_auc"]
].mean().round(3)

test_accuracy    0.847
test_recall      0.784
test_roc_auc     0.906
dtype: float64

In [15]:
from sklearn.tree import DecisionTreeClassifier

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ))
])

In [16]:
tree_scores = cross_validate(
    tree_model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "recall", "roc_auc"]
)

pd.DataFrame(tree_scores)[
    ["test_accuracy", "test_recall", "test_roc_auc"]
].mean().round(3)

test_accuracy    0.773
test_recall      0.729
test_roc_auc     0.832
dtype: float64

In [17]:
from sklearn.ensemble import RandomForestClassifier

forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

In [18]:
forest_scores = cross_validate(
    forest_model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "recall", "roc_auc"]
)

pd.DataFrame(forest_scores)[
    ["test_accuracy", "test_recall", "test_roc_auc"]
].mean().round(3)

test_accuracy    0.814
test_recall      0.774
test_roc_auc     0.900
dtype: float64

In [19]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])

In [20]:
knn_scores = cross_validate(
    knn_model,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "recall", "roc_auc"]
)

pd.DataFrame(knn_scores)[
    ["test_accuracy", "test_recall", "test_roc_auc"]
].mean().round(3)

test_accuracy    0.826
test_recall      0.792
test_roc_auc     0.873
dtype: float64

In [21]:
model_scores = {
    "Baseline": baseline_scores,
    "Logistic Regression": logistic_scores,
    "Decision Tree": tree_scores,
    "Random Forest": forest_scores,
    "KNN": knn_scores
}

comparison = pd.DataFrame({
    name: {
        "Accuracy": scores["test_accuracy"].mean(),
        "Recall": scores["test_recall"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    }
    for name, scores in model_scores.items()
}).T

comparison.round(3)

,Accuracy,Recall,ROC-AUC
Baseline,0.541,0.000,0.500
Logistic Regression,0.847,0.784,0.906
Decision Tree,0.773,0.729,0.832
Random Forest,0.814,0.774,0.900
KNN,0.826,0.792,0.873


In [22]:
fold_variation = pd.DataFrame({
    name: {
        "ROC-AUC mean": scores["test_roc_auc"].mean(),
        "ROC-AUC std": scores["test_roc_auc"].std(ddof=1),
        "Recall mean": scores["test_recall"].mean(),
        "Recall std": scores["test_recall"].std(ddof=1)
    }
    for name, scores in model_scores.items()
}).T

fold_variation.round(3)

,ROC-AUC mean,ROC-AUC std,Recall mean,Recall std
Baseline,0.500,0.000,0.000,0.000
Logistic Regression,0.906,0.022,0.784,0.038
Decision Tree,0.832,0.033,0.729,0.108
Random Forest,0.900,0.034,0.774,0.092
KNN,0.873,0.062,0.792,0.077


## Model selection

Logistic Regression was selected using five-fold cross-validation
on the training set.

- It achieved the highest mean accuracy (0.847) and ROC-AUC (0.906).
- Its mean recall was 0.784.
- It had the lowest variation in ROC-AUC and recall among the learned models.
- KNN had slightly higher mean recall (0.792), but greater variation.
- These comparisons do not establish statistically significant differences.
- The reserved test set was not used to select the model.

In [23]:
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained on", len(X_train), "records.")

Logistic Regression trained on 242 records.


In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)

y_pred = logistic_model.predict(X_test)
y_probability = logistic_model.predict_proba(X_test)[:, 1]

test_results = pd.Series({
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_probability)
})

test_results.round(3)

Accuracy     0.869
Precision    0.812
Recall       0.929
ROC-AUC      0.958
dtype: float64

In [25]:
from sklearn.metrics import confusion_matrix

pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=["Actual absent", "Actual present"],
    columns=["Predicted absent", "Predicted present"]
)

,Predicted absent,Predicted present
Actual absent,27,6
Actual present,2,26


## Final test results

The selected Logistic Regression pipeline was trained on 242 records
and evaluated on 61 reserved test records.

- Accuracy: 86.9%
- Precision: 81.2%
- Recall: 92.9%
- ROC-AUC: 0.958
- True negatives: 27
- False positives: 6
- False negatives: 2
- True positives: 26

These results describe one small test split. Exploratory analysis
previously examined the full dataset, so this holdout is not a completely
independent confirmation. External validation would be needed to assess
generalisability. This educational model is not a clinical diagnostic tool.